# BUSI Notebook 2/3: Per-Class Metrics + ROC/AUC + Model Complexity

**Run Notebook 1 first.** This notebook needs:
1. The **same dataset zip** you used in Notebook 1 (to rebuild the identical test split)
2. The three checkpoint files from Notebook 1's `outputs/` folder:
   `best_multitask_model.pt`, `best_baseline_unet.pt`, `best_baseline_resnet.pt`

This notebook does **not** train anything — it just loads the trained
models and evaluates them more thoroughly. Should take a couple of
minutes on GPU.


## 1. Setup

In [ ]:
!pip install -q torch torchvision pillow numpy matplotlib seaborn scikit-learn

In [ ]:
import os, re, json, zipfile, random, time
from glob import glob
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as tvm
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)
os.makedirs("outputs", exist_ok=True)


## 2. Load the dataset

Upload the **same** BUSI dataset zip you used in Notebook 1, so the
test split matches exactly.


In [ ]:
DATA_ZIP_PATH = "archive.zip"
EXTRACT_DIR = "busi_data"

if not os.path.exists(DATA_ZIP_PATH):
    try:
        from google.colab import files
        print(f"'{DATA_ZIP_PATH}' not found. Please upload the BUSI dataset zip file:")
        uploaded = files.upload()
        DATA_ZIP_PATH = list(uploaded.keys())[0]
    except ImportError:
        raise FileNotFoundError(f"'{DATA_ZIP_PATH}' not found and not running in Colab.")

if not os.path.exists(EXTRACT_DIR):
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    with zipfile.ZipFile(DATA_ZIP_PATH, "r") as zf:
        zf.extractall(EXTRACT_DIR)
    print(f"Extracted to {EXTRACT_DIR}/")

candidates = glob(os.path.join(EXTRACT_DIR, "**", "Dataset_BUSI_with_GT"), recursive=True)
if not candidates:
    if all(os.path.isdir(os.path.join(EXTRACT_DIR, c)) for c in ["benign", "malignant", "normal"]):
        DATA_ROOT = EXTRACT_DIR
    else:
        raise FileNotFoundError("Could not locate Dataset_BUSI_with_GT folder inside the extracted zip.")
else:
    DATA_ROOT = candidates[0]

print("Dataset root:", DATA_ROOT)


## 3. Rebuild the identical index + stratified split (same seed as Notebook 1)

In [ ]:
CLASS_NAMES = ["benign", "malignant", "normal"]
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}

def index_dataset(root_dir):
    records = []
    for cls in CLASS_NAMES:
        cls_dir = os.path.join(root_dir, cls)
        all_files = os.listdir(cls_dir)
        image_files = sorted(f for f in all_files if "_mask" not in f)
        for img_file in image_files:
            stem = os.path.splitext(img_file)[0]
            mask_pattern = re.compile(rf"^{re.escape(stem)}_mask(_\d+)?\.png$")
            masks = sorted(os.path.join(cls_dir, f) for f in all_files if mask_pattern.match(f))
            records.append({"image": os.path.join(cls_dir, img_file), "masks": masks,
                             "label": CLASS_TO_IDX[cls], "class": cls})
    return records

def stratified_split(records, val_frac=0.15, test_frac=0.15, seed=SEED):
    rng = np.random.RandomState(seed)
    by_class = defaultdict(list)
    for r in records:
        by_class[r["class"]].append(r)
    train, val, test = [], [], []
    for cls, items in by_class.items():
        items = items.copy()
        rng.shuffle(items)
        n = len(items)
        n_val = int(round(n * val_frac))
        n_test = int(round(n * test_frac))
        test.extend(items[:n_test])
        val.extend(items[n_test:n_test + n_val])
        train.extend(items[n_test + n_val:])
    rng.shuffle(train); rng.shuffle(val); rng.shuffle(test)
    return train, val, test

records = index_dataset(DATA_ROOT)
train_recs, val_recs, test_recs = stratified_split(records)
print(f"Test set: {len(test_recs)} images")

# Sanity check against Notebook 1's saved split, if available
if os.path.exists("outputs/splits.json"):
    with open("outputs/splits.json") as f:
        saved = json.load(f)
    match = set(r["image"] for r in test_recs) == set(saved["test"])
    print("Test split matches Notebook 1's saved split:", match)


## 4. Test-set Dataset/DataLoader

In [ ]:
IMG_SIZE = 256
BATCH_SIZE = 16

def load_merged_mask(mask_paths, size=None):
    if len(mask_paths) == 0:
        return None
    from PIL import Image
    merged = None
    for p in mask_paths:
        m = np.array(Image.open(p).convert("L"))
        m = (m > 127).astype(np.uint8)
        merged = m if merged is None else np.maximum(merged, m)
    mask_img = Image.fromarray(merged * 255)
    if size is not None:
        mask_img = mask_img.resize(size, resample=Image.NEAREST)
    return mask_img

class BUSIDataset(Dataset):
    def __init__(self, records, img_size=256):
        self.records = records
        self.img_size = img_size

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        from PIL import Image
        rec = self.records[idx]
        size = (self.img_size, self.img_size)
        image = Image.open(rec["image"]).convert("RGB").resize(size, resample=Image.BILINEAR)
        mask_img = load_merged_mask(rec["masks"], size=size)
        if mask_img is None:
            mask_img = Image.new("L", size, 0)
        image_t = TF.to_tensor(image)
        mask_t = (TF.to_tensor(mask_img) > 0.5).float()
        label_t = torch.tensor(rec["label"], dtype=torch.long)
        return image_t, mask_t, label_t

test_ds = BUSIDataset(test_recs, img_size=IMG_SIZE)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print("Test batches:", len(test_loader))


## 5. Rebuild model architectures (must match Notebook 1 exactly)

In [ ]:
class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden = max(channels // reduction, 8)
        self.mlp = nn.Sequential(nn.Linear(channels, hidden), nn.ReLU(inplace=True), nn.Linear(hidden, channels))

    def forward(self, x):
        avg = F.adaptive_avg_pool2d(x, 1).flatten(1)
        mx = F.adaptive_max_pool2d(x, 1).flatten(1)
        attn = torch.sigmoid(self.mlp(avg) + self.mlp(mx))
        return x * attn.unsqueeze(-1).unsqueeze(-1)

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)

    def forward(self, x):
        avg = x.mean(dim=1, keepdim=True)
        mx, _ = x.max(dim=1, keepdim=True)
        attn = torch.sigmoid(self.conv(torch.cat([avg, mx], dim=1)))
        return x * attn

class CBAM(nn.Module):
    def __init__(self, channels, reduction=16, kernel_size=7):
        super().__init__()
        self.channel_attn = ChannelAttention(channels, reduction)
        self.spatial_attn = SpatialAttention(kernel_size)

    def forward(self, x):
        return self.spatial_attn(self.channel_attn(x))

class DecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.upsample = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels + skip_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
        )
        self.attn = CBAM(out_channels)

    def forward(self, x, skip):
        x = self.upsample(x)
        if skip is not None:
            if x.shape[-2:] != skip.shape[-2:]:
                x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
            x = torch.cat([x, skip], dim=1)
        x = self.conv(x)
        return self.attn(x)

class BUSIMultiTaskNet(nn.Module):
    def __init__(self, num_classes=3, pretrained=False):
        super().__init__()
        weights = tvm.ResNet34_Weights.IMAGENET1K_V1 if pretrained else None
        backbone = tvm.resnet34(weights=weights)
        self.stem = nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu)
        self.pool = backbone.maxpool
        self.enc1, self.enc2, self.enc3, self.enc4 = backbone.layer1, backbone.layer2, backbone.layer3, backbone.layer4
        self.dec4 = DecoderBlock(512, 256, 256)
        self.dec3 = DecoderBlock(256, 128, 128)
        self.dec2 = DecoderBlock(128, 64, 64)
        self.dec1 = DecoderBlock(64, 64, 32)
        self.final_upsample = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.seg_head = nn.Conv2d(32, 1, kernel_size=1)
        self.cls_dropout = nn.Dropout(0.3)
        self.cls_fc = nn.Linear(512, num_classes)

    def forward(self, x):
        input_size = x.shape[-2:]
        s0 = self.stem(x); p0 = self.pool(s0)
        s1 = self.enc1(p0); s2 = self.enc2(s1); s3 = self.enc3(s2); s4 = self.enc4(s3)
        d4 = self.dec4(s4, s3); d3 = self.dec3(d4, s2); d2 = self.dec2(d3, s1); d1 = self.dec1(d2, s0)
        d0 = self.final_upsample(d1)
        if d0.shape[-2:] != input_size:
            d0 = F.interpolate(d0, size=input_size, mode="bilinear", align_corners=False)
        seg_logits = self.seg_head(d0)
        mask_prob = torch.sigmoid(seg_logits.detach())
        mask_small = F.adaptive_avg_pool2d(mask_prob, s4.shape[-2:])
        weighted_feats = s4 * (1.0 + mask_small)
        pooled = F.adaptive_avg_pool2d(weighted_feats, 1).flatten(1)
        cls_logits = self.cls_fc(self.cls_dropout(pooled))
        return seg_logits, cls_logits


class BaselineUNet(nn.Module):
    def __init__(self, pretrained=False):
        super().__init__()
        weights = tvm.ResNet34_Weights.IMAGENET1K_V1 if pretrained else None
        backbone = tvm.resnet34(weights=weights)
        self.stem = nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu)
        self.pool = backbone.maxpool
        self.enc1, self.enc2, self.enc3, self.enc4 = backbone.layer1, backbone.layer2, backbone.layer3, backbone.layer4

        def plain_block(in_c, skip_c, out_c):
            return nn.Sequential(nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)), nn.Sequential(
                nn.Conv2d(in_c + skip_c, out_c, 3, padding=1, bias=False),
                nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
                nn.Conv2d(out_c, out_c, 3, padding=1, bias=False),
                nn.BatchNorm2d(out_c), nn.ReLU(inplace=True))
        self.up4, self.conv4 = plain_block(512, 256, 256)
        self.up3, self.conv3 = plain_block(256, 128, 128)
        self.up2, self.conv2 = plain_block(128, 64, 64)
        self.up1, self.conv1 = plain_block(64, 64, 32)
        self.final_upsample = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.seg_head = nn.Conv2d(32, 1, kernel_size=1)

    def _dec(self, up, conv, x, skip):
        x = up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        return conv(torch.cat([x, skip], dim=1))

    def forward(self, x):
        input_size = x.shape[-2:]
        s0 = self.stem(x); p0 = self.pool(s0)
        s1 = self.enc1(p0); s2 = self.enc2(s1); s3 = self.enc3(s2); s4 = self.enc4(s3)
        d4 = self._dec(self.up4, self.conv4, s4, s3)
        d3 = self._dec(self.up3, self.conv3, d4, s2)
        d2 = self._dec(self.up2, self.conv2, d3, s1)
        d1 = self._dec(self.up1, self.conv1, d2, s0)
        d0 = self.final_upsample(d1)
        if d0.shape[-2:] != input_size:
            d0 = F.interpolate(d0, size=input_size, mode="bilinear", align_corners=False)
        return self.seg_head(d0)


class BaselineResNetClassifier(nn.Module):
    def __init__(self, num_classes=3, pretrained=False):
        super().__init__()
        weights = tvm.ResNet34_Weights.IMAGENET1K_V1 if pretrained else None
        self.backbone = tvm.resnet34(weights=weights)
        self.backbone.fc = nn.Linear(self.backbone.fc.in_features, num_classes)

    def forward(self, x):
        return self.backbone(x)


## 6. Load trained checkpoints

Upload `best_multitask_model.pt`, `best_baseline_unet.pt`, and
`best_baseline_resnet.pt` from Notebook 1's `outputs/` folder.


In [ ]:
def ensure_checkpoint(filename):
    if not os.path.exists(filename):
        try:
            from google.colab import files
            print(f"'{filename}' not found. Please upload it:")
            uploaded = files.upload()
            uploaded_name = list(uploaded.keys())[0]
            if uploaded_name != filename:
                os.rename(uploaded_name, filename)
        except ImportError:
            raise FileNotFoundError(f"'{filename}' not found. Place it in the working directory.")
    return filename

mt_ckpt = ensure_checkpoint("best_multitask_model.pt")
unet_ckpt = ensure_checkpoint("best_baseline_unet.pt")
resnet_ckpt = ensure_checkpoint("best_baseline_resnet.pt")

model = BUSIMultiTaskNet(num_classes=len(CLASS_NAMES), pretrained=False).to(DEVICE)
model.load_state_dict(torch.load(mt_ckpt, map_location=DEVICE))
model.eval()

baseline_unet = BaselineUNet(pretrained=False).to(DEVICE)
baseline_unet.load_state_dict(torch.load(unet_ckpt, map_location=DEVICE))
baseline_unet.eval()

baseline_resnet = BaselineResNetClassifier(num_classes=len(CLASS_NAMES), pretrained=False).to(DEVICE)
baseline_resnet.load_state_dict(torch.load(resnet_ckpt, map_location=DEVICE))
baseline_resnet.eval()

print("All checkpoints loaded.")


## 7. Run inference on the test set (collect predictions + probabilities)

In [ ]:
def dice_score(logits, targets, threshold=0.5, eps=1e-6):
    with torch.no_grad():
        preds = (torch.sigmoid(logits) > threshold).float().flatten(1)
        targets = targets.flatten(1)
        intersection = (preds * targets).sum(dim=1)
        union = preds.sum(dim=1) + targets.sum(dim=1)
        return ((2 * intersection + eps) / (union + eps)).mean().item()

def iou_score(logits, targets, threshold=0.5, eps=1e-6):
    with torch.no_grad():
        preds = (torch.sigmoid(logits) > threshold).float().flatten(1)
        targets = targets.flatten(1)
        intersection = (preds * targets).sum(dim=1)
        union = preds.sum(dim=1) + targets.sum(dim=1) - intersection
        return ((intersection + eps) / (union + eps)).mean().item()

all_labels, all_preds_mt, all_preds_resnet = [], [], []
all_probs_mt = []
dice_mt, iou_mt = [], []

with torch.no_grad():
    for images, masks, labels in test_loader:
        images, masks, labels = images.to(DEVICE), masks.to(DEVICE), labels.to(DEVICE)
        seg_logits, cls_logits = model(images)
        dice_mt.append(dice_score(seg_logits, masks))
        iou_mt.append(iou_score(seg_logits, masks))
        all_preds_mt.extend(cls_logits.argmax(dim=1).cpu().numpy())
        all_probs_mt.extend(torch.softmax(cls_logits, dim=1).cpu().numpy())

        resnet_logits = baseline_resnet(images)
        all_preds_resnet.extend(resnet_logits.argmax(dim=1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

all_labels = np.array(all_labels)
all_preds_mt = np.array(all_preds_mt)
all_preds_resnet = np.array(all_preds_resnet)
all_probs_mt = np.array(all_probs_mt)

print(f"Multi-task model -- test Dice: {np.mean(dice_mt):.4f}  IoU: {np.mean(iou_mt):.4f}  "
      f"Accuracy: {np.mean(all_preds_mt == all_labels):.4f}")


## 8. Per-class metrics (Precision, Recall, F1 per class)

In [ ]:
report_dict = classification_report(all_labels, all_preds_mt, target_names=CLASS_NAMES,
                                     output_dict=True, zero_division=0)
report_text = classification_report(all_labels, all_preds_mt, target_names=CLASS_NAMES, zero_division=0)
print("Per-class metrics -- Multi-Task Model (test set)")
print(report_text)

with open("outputs/per_class_metrics.json", "w") as f:
    json.dump(report_dict, f, indent=2)


## 9. ROC curves and AUC (one-vs-rest, multi-class)

In [ ]:
y_true_bin = label_binarize(all_labels, classes=list(range(len(CLASS_NAMES))))

fpr, tpr, roc_auc = {}, {}, {}
for i, cls_name in enumerate(CLASS_NAMES):
    fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], all_probs_mt[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

fpr["micro"], tpr["micro"], _ = roc_curve(y_true_bin.ravel(), all_probs_mt.ravel())
roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

plt.figure(figsize=(6, 6))
colors = ["#1f77b4", "#d62728", "#2ca02c"]
for i, cls_name in enumerate(CLASS_NAMES):
    plt.plot(fpr[i], tpr[i], color=colors[i], lw=2, label=f"{cls_name} (AUC = {roc_auc[i]:.3f})")
plt.plot(fpr["micro"], tpr["micro"], color="black", linestyle="--", lw=2,
          label=f"micro-average (AUC = {roc_auc['micro']:.3f})")
plt.plot([0, 1], [0, 1], color="gray", lw=1, linestyle=":")
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("ROC Curves — Multi-Task Model (Test Set)")
plt.legend(loc="lower right")
plt.tight_layout(); plt.savefig("outputs/roc_curves.png", dpi=150); plt.show()

print("AUC per class:", {CLASS_NAMES[i]: round(roc_auc[i], 4) for i in range(len(CLASS_NAMES))})
print("Micro-average AUC:", round(roc_auc["micro"], 4))


## 10. Model complexity comparison

Parameter counts and inference speed — useful if you frame the
multi-task model as an efficient alternative to running two separate
networks.


In [ ]:
def count_params(m):
    return sum(p.numel() for p in m.parameters())

def measure_inference_time(m, input_size=(1, 3, IMG_SIZE, IMG_SIZE), n_runs=30, device=DEVICE):
    m.eval()
    dummy = torch.randn(*input_size).to(device)
    with torch.no_grad():
        for _ in range(5):
            m(dummy)
    if device.type == "cuda":
        torch.cuda.synchronize()
    start = time.time()
    with torch.no_grad():
        for _ in range(n_runs):
            m(dummy)
    if device.type == "cuda":
        torch.cuda.synchronize()
    elapsed = time.time() - start
    return (elapsed / n_runs) * 1000

complexity_rows = []
for name, m in [("Multi-Task Model", model),
                ("Baseline U-Net (seg only)", baseline_unet),
                ("Baseline ResNet34 (cls only)", baseline_resnet)]:
    n_params = count_params(m)
    t_ms = measure_inference_time(m)
    complexity_rows.append((name, n_params, t_ms))

print(f"{'Model':<30}{'Params':<15}{'Inference (ms/img)':<20}")
print("-" * 65)
for name, n_params, t_ms in complexity_rows:
    params_str = f"{n_params:,}"
    print(f"{name:<30}{params_str:<15}{t_ms:<20.2f}")

mt_params = complexity_rows[0][1]
combined_baseline_params = complexity_rows[1][1] + complexity_rows[2][1]
print(f"\nMulti-task params: {mt_params:,}")
print(f"Combined baseline params (U-Net + ResNet separately): {combined_baseline_params:,}")
print(f"Parameter savings: {(1 - mt_params / combined_baseline_params) * 100:.1f}%")

with open("outputs/model_complexity.json", "w") as f:
    json.dump({name: {"params": n_params, "inference_ms": t_ms}
               for name, n_params, t_ms in complexity_rows}, f, indent=2)


## Done — Notebook 2

Files saved to `outputs/`:
- `per_class_metrics.json`
- `roc_curves.png`
- `model_complexity.json`

**Next:** open **Notebook 3** for the ablation study and the
statistical significance test.
